# Consistent Stock Selection Portfolio

## Goal

This notebook help evaluate if a stock that the model consistently selects performs better then the S&P 500.

While the Random Forest model had performance better than random, it did not have enough to have returns better than S&P 500. This notebook focuses on stocks with repeated high probabilities in order to find more consistently good stocks.

## 1. Load Model Predictions

The classifier output contains:

* Ticker
* Date of Window
* Model Probabilities
* Model Predictions
* Data features
* Weather the stock beat the S&P 500

In [7]:
# Importing File
import pandas as pd
import numpy as np
import yfinance as yf

# Parameters
SCORED_FILE = "../logs/classification/test_scored_gen1.parquet"

TOP_N = 10
MIN_SELECTIONS = 15
PORTFOLIO_SIZE = 20

df = pd.read_parquet( SCORED_FILE )

df = df.reset_index()

# Test print of (Rows, Columns)
df.shape

(147236, 122)

## 2. Identify Top Model Picks

For each predition date, the stocks with the highest model confience are selected. These represent the stocks the model would have purchased at each time window.

In [8]:
logreg = df.sort_values( ["date", "logreg_probability"], ascending=[True, False] ).groupby("date").head(10)

logreg[["ticker", "date", "logreg_probability", "future_ret_5y"]]

,ticker,date,logreg_probability,future_ret_5y
68304,ENPH,2020-07-06,0.509219,-0.183214
110082,TPL,2020-07-06,0.463234,5.067707
16929,NOA,2020-07-06,0.432630,1.768323
2844,GPRK,2020-07-06,0.430606,-0.207104
1977,CWST,2020-07-06,0.426460,1.150096
...,...,...,...,...
33288,DHX,2021-07-01,0.700836,0.104651
21562,CALX,2021-07-01,0.698702,-0.208155
93305,BEEM,2021-07-01,0.698276,-0.963590
46666,UPBD,2021-07-01,0.697807,-0.492780


In [9]:
# Get top N highest RF probabilities
top_picks = (
    df.sort_values(
        ["date", "logreg_probability"],
        ascending=[True, False]
    )
    .groupby("date")
    .head(TOP_N)
)

top_picks.head()

,index,date,ticker,price,beta_1y,alpha_1y,sector,quote_type,log_market_cap,pe,...,risk_adjusted_5y,sector_is_strong,sector_is_trending,sector_high_breadth,quality_score,beats_market_5y,logreg_probability,logreg_prediction,rf_probability,rf_prediction
68304,68304,2020-07-06,ENPH,50.160000,2.051012,0.015733,Technology,EQUITY,22.612001,44.163149,...,1.019746,True,False,False,2,0,0.509219,1,0.391498,0
110082,110082,2020-07-06,TPL,58.611343,1.702131,-0.021840,Energy,EQUITY,22.120175,8.397506,...,4.200884,False,False,False,2,1,0.463234,0,0.321838,0
16929,16929,2020-07-06,NOA,5.908278,1.475565,0.030825,Energy,EQUITY,18.891494,7.934530,...,2.188485,False,False,False,0,1,0.432630,0,0.317695,0
2844,2844,2020-07-06,GPRK,8.852048,1.626595,-0.010458,Energy,EQUITY,20.168942,9.529520,...,1.740252,False,False,False,0,0,0.430606,0,0.308575,0
1977,1977,2020-07-06,CWST,51.900002,1.216883,-0.006451,Industrials,EQUITY,21.901942,439.707108,...,5.753195,False,False,False,3,1,0.426460,0,0.355733,0


## 3. Measure Stock Consistency

How often a stock is selected is then counted. Maximum for this is 51, as there are 51 possible time windows.

Stocks selected repeatedly represent stronger model conviction.

In [10]:
# Sort stocks by number of times selected
consistent = (
    top_picks
    .groupby("ticker")
    .agg(
        times_selected=("ticker", "count"),
        avg_probability=("logreg_probability", "mean"),
        median_probability=("logreg_probability", "median"),
        avg_future_return=("future_excess_5y", "mean"),
        median_future_return=("future_excess_5y", "median"),
        beat_rate=("beats_market_5y", "mean")
    )
    .sort_values(
        "times_selected",
        ascending=False
    )
)

consistent.head(50)

,times_selected,avg_probability,median_probability,avg_future_return,median_future_return,beat_rate
ticker,,,,,,
TKO,29,0.733240,0.760876,2.642895,2.782483,1.000000
MCS,28,0.747230,0.757713,-0.643680,-0.728417,0.035714
SBGI,20,0.748007,0.779562,-1.240855,-1.272713,0.000000
IRDM,20,0.731409,0.752621,-1.249456,-1.412716,0.000000
ENPH,18,0.650249,0.668606,-1.588082,-1.567529,0.000000
TZOO,17,0.798009,0.842640,-1.214803,-1.314709,0.000000
QNST,17,0.691820,0.632205,-1.204883,-1.223724,0.000000
GAIA,16,0.725588,0.709063,-1.595959,-1.586199,0.000000
CNK,15,0.657796,0.611764,0.234555,0.224653,0.533333


## 4. Filter stocks below threshold

A list of stocks have have been selected at or above the threshold are selected. Stocks only selected by the algorithm a few times are discarded as inconsistent.

In [11]:
# Filter to stocks have have been selected at, or above, limit
consistent_stocks = (
    consistent[
        consistent["times_selected"] >= MIN_SELECTIONS
    ]
)

consistent_stocks

,times_selected,avg_probability,median_probability,avg_future_return,median_future_return,beat_rate
ticker,,,,,,
TKO,29,0.733240,0.760876,2.642895,2.782483,1.000000
MCS,28,0.747230,0.757713,-0.643680,-0.728417,0.035714
SBGI,20,0.748007,0.779562,-1.240855,-1.272713,0.000000
IRDM,20,0.731409,0.752621,-1.249456,-1.412716,0.000000
ENPH,18,0.650249,0.668606,-1.588082,-1.567529,0.000000
TZOO,17,0.798009,0.842640,-1.214803,-1.314709,0.000000
QNST,17,0.691820,0.632205,-1.204883,-1.223724,0.000000
GAIA,16,0.725588,0.709063,-1.595959,-1.586199,0.000000
CNK,15,0.657796,0.611764,0.234555,0.224653,0.533333


## 5. Limit number of stocks selected

The number of consistently selected stocks are capped to ensure both selection of the best results as well as keeping a simple portfolio.

In [12]:
# Filter stocks down to maximum number
portfolio = (
    consistent_stocks
    .sort_values(
        [
            "times_selected",
            "avg_probability"
        ],
        ascending=False
    )
    .head(PORTFOLIO_SIZE)
)

portfolio[
    [
        "times_selected",
        "avg_probability",
        "avg_future_return",
        "median_future_return",
        "beat_rate"
    ]
]

,times_selected,avg_probability,avg_future_return,median_future_return,beat_rate
ticker,,,,,
TKO,29,0.733240,2.642895,2.782483,1.000000
MCS,28,0.747230,-0.643680,-0.728417,0.035714
SBGI,20,0.748007,-1.240855,-1.272713,0.000000
IRDM,20,0.731409,-1.249456,-1.412716,0.000000
ENPH,18,0.650249,-1.588082,-1.567529,0.000000
TZOO,17,0.798009,-1.214803,-1.314709,0.000000
QNST,17,0.691820,-1.204883,-1.223724,0.000000
GAIA,16,0.725588,-1.595959,-1.586199,0.000000
CNET,15,0.713594,-1.972160,-1.945903,0.000000


In [13]:
# Most current window of data
last_date = df["date"].max()

print(last_date)

2021-07-01 00:00:00


## 6. Generate Portfolio List

A list of the top stocks.

This could be expanded to include features of the stocks, either for the most current window or an average over all windows.

In [14]:
#Tickers for all stocks in portfolio
holdings = portfolio.index.tolist()

holdings

['TKO', 'MCS', 'SBGI', 'IRDM', 'ENPH', 'TZOO', 'QNST', 'GAIA', 'CNET', 'CNK']

In [15]:
buy_and_hold = df[
    (df["date"] == last_date) &
    (df["ticker"].isin(holdings))
].copy()

buy_and_hold[
    [
        "ticker",
        "rf_probability",
        "future_excess_5y",
        "beats_market_5y"
    ]
]

,ticker,rf_probability,future_excess_5y,beats_market_5y
28896,TKO,0.236105,1.876774,1
54706,MCS,0.167140,-0.690306,0
68354,ENPH,0.310479,-1.597970,0
74336,CNK,0.170069,-0.388549,0
88518,QNST,0.251015,-1.030117,0
92540,IRDM,0.275181,-0.382900,0
103345,GAIA,0.162330,-1.667009,0
110438,TZOO,0.185889,-1.068429,0
126413,SBGI,0.159996,-1.226430,0


In [16]:
spy = yf.download(
    "SPY",
    start=last_date,
    end=pd.Timestamp(last_date) + pd.DateOffset(years=5),
    auto_adjust=True
)

[*********************100%***********************]  1 of 1 completed


# Result Summary

The excess returns (return above SPY) and actual returns are both calculated.

For easier understanding, a simulation is also run, based on a $10,000 investment, equally distributed to each stock.

The results suggest that repeated model selection is useful as an investment signal.

In [17]:
portfolio_return = buy_and_hold["future_ret_5y"].mean()
portfolio_excess = buy_and_hold["future_excess_5y"].mean()

print(
    f"Portfolio excess return: {portfolio_excess:.2%}"
)

print(
    f"Portfolio return: {portfolio_return:.2%}"
)

spy_return = (
    spy["Close"].iloc[-1].item() /
    spy["Close"].iloc[0].item()
    - 1
)

initial_investment = 10000

spy_value = initial_investment * (1 + spy_return)

ending_value = (
    initial_investment *
    (1 + portfolio_return)
)

print(
    "\n",
    "---Portfolio simulation ---",
)

print(
    f"Starting value: ${initial_investment:,.2f}"
)

print(
    f"SPY Ending value: ${spy_value:,.2f}"
)

print(
    f"Model Ending value: ${ending_value:,.2f}"
)

return_difference = ( ending_value - spy_value ) / spy_value

print(
    f"Different compared to SPY: {return_difference:+.2%}"
)

Portfolio excess return: -68.61%
Portfolio return: 16.77%

 ---Portfolio simulation ---
Starting value: $10,000.00
SPY Ending value: $18,562.83
Model Ending value: $11,676.69
Different compared to SPY: -37.10%
